## Topic: Database Integration (Store Predictions + Build Data Pipeline)
###  Learning Objectives

By the end of today, you will:

- Understand how to connect a database with your ML API
- Store:
  - Input data
  - Predictions
- Use SQLite (beginner-friendly database)
- Build a data pipeline (API → DB → Model)

### Step 1 — Why we add a Database

Right now your API does this:
```
User → API → Model → Prediction → Response
```
After response… prediction disappears

Companies need to:

- Track predictions
- Audit model behaviour
- Analyse users later
- Retrain models using stored data

So we add storage:

```
User → API → Model → Prediction → DATABASE
```
This is called a data pipeline.

### Step 2 — Import SQLite

SQLite is perfect for beginners because:

- No server installation
- Just a file (predictions.db)
- Works locally and on cloud

Add this at top of app.py:
```
import sqlite3
```

### Step 3 — Create Database + Table (VERY IMPORTANT)

We must create the database when the app starts.

Add this function above your routes:
```
# ============================================
# DATABASE INITIALIZATION
# ============================================

def init_db():
    conn = sqlite3.connect("predictions.db")
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        feature1 REAL,
        feature2 REAL,
        prediction INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    conn.commit()
    conn.close()

# Run database creation when app starts
init_db()
```
#### Why this matters

- Creates DB file automatically
- Creates table only once
- Adds timestamp column (production practice)

### Step 4 — Create DB Connection Function

We should not repeat connection code everywhere.

Add this:
```
def get_db_connection():
    return sqlite3.connect("predictions.db")
```
This makes code clean and reusable.

### Step 5 — Modify /predict to Store Data

Right now your endpoint only predicts.

We will make it:
1. Predict
2. Save input + prediction into DB
3. Return response

Replace your /predict function with this:
```
@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        if not data or "features" not in data:
            return jsonify({"status":"error","message":"Missing features"}),400

        features = data["features"]

        # Make prediction
        prediction = model.predict([features])[0]

        # -------- STORE IN DATABASE --------
        conn = get_db_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO predictions (feature1, feature2, prediction)
        VALUES (?, ?, ?)
        """, (features[0], features[1], int(prediction)))

        conn.commit()
        conn.close()
        # -----------------------------------

        return jsonify({
            "status":"success",
            "prediction": int(prediction)
        })

    except Exception as e:
        return jsonify({"status":"error","message":str(e)}),500
```
Why we store only feature1 & feature2?

To keep learning simple.

Later you can store all 30 features.

### Step 6 — Test POST Request

Send in Postman:
```
{
  "features": [1,1,0,0,1,0,1,0,1,1,0,0,1,0,1,0,1,0,1,0,50,50000,1,0,1,0,1,0,1,0]
}
```
Every request now gets saved in database

### Step 7 — Create /history API

Now we create endpoint to view stored predictions.

Add this below /predict:
```
@app.route("/history", methods=["GET"])
def history():
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("SELECT * FROM predictions")
    rows = cursor.fetchall()

    conn.close()

    return jsonify(rows)
```

###  Final Project Folder Structure

Your project should now look exactly like this:
```
ChurnPredictionAPI/
│
├── app.py                ← main Flask app
├── predictions.db        ← created automatically after first run
├── requirements.txt
├── runtime.txt
├── Procfile
│
└── model/
    └── churn_model.pkl
```
### What each file does
# Project Files Overview

| File                | Purpose                                      |
|---------------------|----------------------------------------------|
| `app.py`            | Flask API + ML + Database                    |
| `churn_model.pkl`   | Trained ML model                             |
| `predictions.db`    | Stores prediction history                    |
| `requirements.txt`  | Packages for deployment                      |
| `Procfile`          | Tells Render how to run app                  |
| `runtime.txt`       | Python version                               |

### Where your code should go in app.py (order matters)

Your file must follow this order:

1. Imports
2. Create Flask app
3. Load model
4. Initialize database
5. Routes (/, /api, /predict, /history)
6. Run app

Here is the clean combined version showing correct placement:
```
# ============================================
# IMPORTS
# ============================================
from flask import Flask, request, jsonify
import joblib
import sqlite3

# ============================================
# CREATE APP + LOAD MODEL
# ============================================
app = Flask(__name__)
model = joblib.load("model/churn_model.pkl")

# ============================================
# DATABASE INITIALIZATION
# ============================================
def init_db():
    conn = sqlite3.connect("predictions.db")
    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        feature1 REAL,
        feature2 REAL,
        prediction INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    conn.commit()
    conn.close()

init_db()

def get_db_connection():
    return sqlite3.connect("predictions.db")

# ============================================
# ROUTES
# ============================================

@app.route("/")
def home():
    return jsonify({"message": "Churn Prediction API running 🚀"})

@app.route("/api", methods=["GET"])
def api_status():
    return jsonify({"status":"success","message":"API working"})

# ---------- PREDICTION + STORE IN DB ----------
@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        if not data or "features" not in data:
            return jsonify({"status":"error","message":"Missing features"}),400

        features = data["features"]
        prediction = model.predict([features])[0]

        # STORE IN DATABASE
        conn = get_db_connection()
        cursor = conn.cursor()

        cursor.execute("""
        INSERT INTO predictions (feature1, feature2, prediction)
        VALUES (?, ?, ?)
        """, (features[0], features[1], int(prediction)))

        conn.commit()
        conn.close()

        return jsonify({"status":"success","prediction":int(prediction)})

    except Exception as e:
        return jsonify({"status":"error","message":str(e)}),500

# ---------- VIEW HISTORY ----------
@app.route("/history", methods=["GET"])
def history():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM predictions")
    rows = cursor.fetchall()
    conn.close()

    return jsonify(rows)

# ============================================
# RUN APP
# ============================================
if __name__ == "__main__":
    app.run(debug=True)


### Step 8 — Test History API

Open browser:
```
http://127.0.0.1:5000/history
```
You’ll see something like:
```
[
 [1, 1.0, 1.0, 0, "2026-04-21 12:30:22"],
 [2, 1.0, 1.0, 1, "2026-04-21 12:32:10"]
]
```
Congratulations — you now have prediction history!

### 1. requirements.txt

This file lists Python packages your app needs.

Create a file named requirements.txt and paste:
```
flask
gunicorn
numpy
pandas
scikit-learn
joblib
```
### Why each package?

| Package        | Why needed                     |
|----------------|--------------------------------|
| flask          | Web API                        |
| gunicorn       | Production server (Render requirement) |
| numpy          | Model input                    |
| pandas         | Data processing                |
| scikit-learn   | ML model                       |
| joblib         | Load saved model               |

### 2. runtime.txt

This tells Render which Python version to use.

Create runtime.txt and write exactly one line:
```
python-3.11.9
```
No spaces. No extra lines.

### 3. Procfile (very important)

This tells the cloud how to start your app.

Create file named Procfile
 No extension. Just Procfile.

Paste this line:
```
web: gunicorn app:app
```
# What this means

| Part        | Meaning                                      |
|-------------|----------------------------------------------|
| web         | Start a web service                          |
| gunicorn    | Production server                            |
| app:app     | file `app.py` → variable `app = Flask(__name__)` |

### Step 9 — What You Built Today

You now have:

| Component       | Status |
|----------------|--------|
| Flask API      | ✔      |
| ML Model       | ✔      |
| Database       | ✔      |
| Data Pipeline  | ✔      |

This is a mini production ML system.

### Questions Answers

#### What is SQLite?
Lightweight file-based database.

#### Why DB in ML apps?
To store predictions for tracking & retraining.

#### What is a data pipeline?
Flow: Input → Model → Storage → Analysis.